In [3]:
from copy import deepcopy
import numpy as np
from ip import *

# some hyperparameters
# define how many layers to load into memory
L_MAX          = 64
L              = 64
L_BEGIN        = 0
L_CLOSE        = L
G              = 8
GW             = 128

DW_WQ          = 4
DW_WS          = 3

T_MAX          = 512
T              = 1
C              = 2560

TP             = 1
CP             = G
TT             = T // TP
CT             = C // CP

CI1 = 2560
CI2 = 5120
CO1 = 10576
CO2 = 2560
P = 64

CC = 5376
C2 = 5120
N = 128
NT = N/CP
CCT = CC/CP
C2T = C2/CP
NP = CP

NUM_WQ         = CI1 * CO1 + CI2 * CO2
NUM_WS         = NUM_WQ / GW
NUM_X          = T*C
print(f"NUM_WQ: {NUM_WQ}")
print(f"NUM_WS: {NUM_WS}")
print(f"NUM_X:  {NUM_X}")

TOTAL_CYCS=635199
DW_MAXI = CP * CP * DW_WQ
BYTES_PER_PACK = DW_MAXI / 8
C_CYCS=714
H_CYCS=21760
W_BITS         = NUM_WQ*DW_WQ + NUM_WS*DW_WS*2
W_BYTES        = int(TOTAL_CYCS*BYTES_PER_PACK)
X_BITS         = NUM_X * 32
X_BYTES        = X_BITS // 8
print(f"W_BITS:  {W_BITS}")
print(f"W_BYTES: {W_BYTES}")
print(f"X_BITS:  {X_BITS}")
print(f"X_BYTES: {X_BYTES}")




C_BYTES=int(C_CYCS*BYTES_PER_PACK)
H_BYTES=int(H_CYCS*BYTES_PER_PACK)

NUM_WQ: 40181760
NUM_WS: 313920.0
NUM_X:  2560
W_BITS:  162610560.0
W_BYTES: 20326368
X_BITS:  81920
X_BYTES: 10240


In [4]:
gpio        = AXI_REGISTER  (0xAB00_0000, np.uint32)

ADDR_W      = 0x500_0000_0000
ADDR_X      = 0x500_8000_0000
ADDR_Y      = 0x500_9000_0000
ADDR_C      = 0x500_A000_0000
ADDR_H      = 0x500_B000_0000

lpddr_w     = AXI_MEM       (ADDR_W, dtype=np.byte,  bytes=0x0_8000_0000) # byte view
lpddr_x     = AXI_MEM       (ADDR_X, dtype=np.int32, bytes=X_BYTES      ) # int  view
lpddr_y     = AXI_MEM       (ADDR_Y, dtype=np.int32, bytes=X_BYTES      ) # int  view
lpddr_c     = AXI_MEM       (ADDR_C, dtype=np.byte, bytes=0x0_1000_0000) # int  view
lpddr_h     = AXI_MEM       (ADDR_H, dtype=np.byte, bytes=0x0_1000_0000) # int  view

clock       = AXI_CLOCK     (0xA400_0000)
pl_reset    = PL_RESET      ()

for i in range(4):
    gpio.write(0x0)
    sleep(0.2)
    gpio.write(0xf)
    sleep(0.2)

In [5]:
# address
ADDR_L_BEGIN    = 0x0000
ADDR_L_CLOSE    = 0x0010
ADDR_MEMORY_X   = 0x0020
ADDR_MEMORY_W   = 0x0030
ADDR_MEMORY_Y   = 0x0040
ADDR_POS        = 0x0050
ADDR_T          = 0x0060
ADDR_IDLE       = 0x0070
ADDR_MEMORY_C   = 0x0080
ADDR_MEMORY_H   = 0x0090
# constants

# create IP
mamba           = AXI_IP(0xA401_0000, 
                         [
                                ("L_BEGIN",   ADDR_L_BEGIN,  np.int64),
                                ("L_CLOSE",   ADDR_L_CLOSE,  np.int64),
                                ("MEMORY_X",  ADDR_MEMORY_X, np.int64),
                                ("MEMORY_W",  ADDR_MEMORY_W, np.int64),
                                ("MEMORY_Y",  ADDR_MEMORY_Y, np.int64),
                                ("MEMORY_C",  ADDR_MEMORY_C, np.int64),
                                ("MEMORY_H",  ADDR_MEMORY_H, np.int64),
                                ("POS",       ADDR_POS,      np.int64),
                                ("T",         ADDR_T,        np.int64),
                                ("IDLE",      ADDR_IDLE,     np.int64),
                         ])

In [13]:
# put W into memory
def first_bytes(arr):
    print("First 10 bytes in hex:", ", ".join([f"0x{val:02X}" for val in arr[:10].astype(np.uint8)]))
for l in range(L_BEGIN, L_CLOSE):
    CONDENSED_W = np.fromfile(f"./bin/CONDENSED_W_layer{l}.bin",  dtype=np.byte)
    assert(CONDENSED_W.size == W_BYTES)
    print(f"Layer {l} Read")
    lpddr_w[l*W_BYTES:(l+1)*W_BYTES] = CONDENSED_W
#     print(f"Layer {l} Put")
#     first_bytes(CONDENSED_W)
#     first_bytes(lpddr_w[l*W_BYTES:(l+1)*W_BYTES])
#     print("")

Layer 0 Read
Layer 1 Read
Layer 2 Read
Layer 3 Read
Layer 4 Read
Layer 5 Read
Layer 6 Read
Layer 7 Read
Layer 8 Read
Layer 9 Read
Layer 10 Read
Layer 11 Read
Layer 12 Read
Layer 13 Read
Layer 14 Read
Layer 15 Read
Layer 16 Read
Layer 17 Read
Layer 18 Read
Layer 19 Read
Layer 20 Read
Layer 21 Read
Layer 22 Read
Layer 23 Read
Layer 24 Read
Layer 25 Read
Layer 26 Read
Layer 27 Read
Layer 28 Read
Layer 29 Read
Layer 30 Read
Layer 31 Read
Layer 32 Read
Layer 33 Read
Layer 34 Read
Layer 35 Read
Layer 36 Read
Layer 37 Read
Layer 38 Read
Layer 39 Read
Layer 40 Read
Layer 41 Read
Layer 42 Read
Layer 43 Read
Layer 44 Read
Layer 45 Read
Layer 46 Read
Layer 47 Read
Layer 48 Read
Layer 49 Read
Layer 50 Read
Layer 51 Read
Layer 52 Read
Layer 53 Read
Layer 54 Read
Layer 55 Read
Layer 56 Read
Layer 57 Read
Layer 58 Read
Layer 59 Read
Layer 60 Read
Layer 61 Read
Layer 62 Read
Layer 63 Read


In [14]:
for l in range(L_BEGIN, L_CLOSE):
    CONDENSED_CW = np.fromfile(f"./bin/CONDENSED_CW_layer{l}.bin",  dtype=np.byte)
    assert(CONDENSED_CW.size == C_BYTES)
    print(f"Layer {l} Read")
    lpddr_c[(l*2+1)*C_BYTES:(l*2+2)*C_BYTES] = CONDENSED_CW
#     print(f"Layer {l} Put")
#     first_bytes(CONDENSED_W)
#     first_bytes(lpddr_w[l*W_BYTES:(l+1)*W_BYTES])
#     print("")

Layer 0 Read
Layer 1 Read
Layer 2 Read
Layer 3 Read
Layer 4 Read
Layer 5 Read
Layer 6 Read
Layer 7 Read
Layer 8 Read
Layer 9 Read
Layer 10 Read
Layer 11 Read
Layer 12 Read
Layer 13 Read
Layer 14 Read
Layer 15 Read
Layer 16 Read
Layer 17 Read
Layer 18 Read
Layer 19 Read
Layer 20 Read
Layer 21 Read
Layer 22 Read
Layer 23 Read
Layer 24 Read
Layer 25 Read
Layer 26 Read
Layer 27 Read
Layer 28 Read
Layer 29 Read
Layer 30 Read
Layer 31 Read
Layer 32 Read
Layer 33 Read
Layer 34 Read
Layer 35 Read
Layer 36 Read
Layer 37 Read
Layer 38 Read
Layer 39 Read
Layer 40 Read
Layer 41 Read
Layer 42 Read
Layer 43 Read
Layer 44 Read
Layer 45 Read
Layer 46 Read
Layer 47 Read
Layer 48 Read
Layer 49 Read
Layer 50 Read
Layer 51 Read
Layer 52 Read
Layer 53 Read
Layer 54 Read
Layer 55 Read
Layer 56 Read
Layer 57 Read
Layer 58 Read
Layer 59 Read
Layer 60 Read
Layer 61 Read
Layer 62 Read
Layer 63 Read


In [15]:
# load X
REF_X = np.fromfile(f"./bin/before_rms1_layer{L_BEGIN  }.bin", dtype=np.int64)[:NUM_X]
REF_Y = np.fromfile(f"./bin/output_layer{L_CLOSE-1}.bin", dtype=np.int64)[:NUM_X]
# put X into memory
lpddr_x[:NUM_X] = REF_X
print("X ", REF_X[:100])
# lpddr_x[:NUM_X] = REF_X
print("LPDDR X  ", lpddr_x[:100])
# initialize Y to 0
lpddr_y[:] = 0
print(lpddr_y[:100])

X  [ 195  103   94   14 -125  -80  378  -37  -85   84   43   19  115 -197
 -120  -80  -13 -139 -111  -38   95 -180   -7  -26 -119  -88   90 -266
  287   58 -206 -188  -17 -147  -10  226   35  -88 -357 -154   30  -58
   36   -3 -113   42 -208 -166   43 -257  -61    0   -9 -144  -27   63
  -55  -35   89  -85  126  204 -227  -72  -22  243 -226  -57  157   46
   99  -38  112 -148  -62   94   10   -6  -90 -172 -116  -93  353  156
   65  -72 -228   -1 -124 -264  342  115 -225  -53  -95 -166   42   53
   69  138]
LPDDR X   [ 195  103   94   14 -125  -80  378  -37  -85   84   43   19  115 -197
 -120  -80  -13 -139 -111  -38   95 -180   -7  -26 -119  -88   90 -266
  287   58 -206 -188  -17 -147  -10  226   35  -88 -357 -154   30  -58
   36   -3 -113   42 -208 -166   43 -257  -61    0   -9 -144  -27   63
  -55  -35   89  -85  126  204 -227  -72  -22  243 -226  -57  157   46
   99  -38  112 -148  -62   94   10   -6  -90 -172 -116  -93  353  156
   65  -72 -228   -1 -124 -264  342  115 -225  -53  

In [7]:
def run_mamba(_L_BEGIN, _L_CLOSE, _POS, freq=MHz400, verbose=False):
    pl_reset.reset()
    clock.refresh(freq)
    # try reading idle
    print(f"Reading IDLE: {mamba.IDLE}")
    # start
    mamba.L_BEGIN  = _L_BEGIN
    mamba.L_CLOSE  = _L_CLOSE
    mamba.MEMORY_W = ADDR_W
    mamba.MEMORY_X = ADDR_X
    mamba.MEMORY_Y = ADDR_Y
    mamba.MEMORY_C = ADDR_C
    mamba.MEMORY_H = ADDR_H
    mamba.POS      = _POS
    if verbose:
        print(f"L_BEGIN:  {mamba.L_BEGIN  }")
        print(f"L_CLOSE:  {mamba.L_CLOSE  }")
        print(f"MEMORY_W: {mamba.MEMORY_W :x}")
        print(f"MEMORY_X: {mamba.MEMORY_X :x}")
        print(f"MEMORY_Y: {mamba.MEMORY_Y :x}")
        print(f"MEMORY_C: {mamba.MEMORY_C :x}")
        print(f"MEMORY_H: {mamba.MEMORY_H :x}")
        print(f"POS:      {mamba.POS      }")

    sleep(0.1) # wait the daisy chain to propagate
    mamba.T = 1

    # time
    start   = time.time()
    while True:
        sleep(0.0001)
        if mamba.IDLE:
            break
    end     = time.time()
    elapsed = end - start

    print(f"elapsed time:       {elapsed: .4f}")

#     return deepcopy(lpddr_y[:NUM_X])

# in the simulation result, we can have a higher result, 3.07 token/s

In [8]:
def clear_state(_L_BEGIN, _L_CLOSE,_lpddr_c,_lpddr_h):
    CONDENSED_C=np.zeros(C_BYTES,dtype=np.byte)
    CONDENSED_H=np.zeros(H_BYTES,dtype=np.byte)
    for l in range(_L_BEGIN, _L_CLOSE):
#         print(f"Layer {l} Read")
        _lpddr_c[l*2*C_BYTES:(l*2+1)*C_BYTES] = CONDENSED_C
        _lpddr_h[l*H_BYTES:(l+1)*H_BYTES] = CONDENSED_H
#         print(f"Layer {l} Put")
#         first_bytes(lpddr_c[l*C_BYTES:(l+1)*C_BYTES])
#         first_bytes(lpddr_h[l*H_BYTES:(l+1)*H_BYTES])
#         print("")

In [16]:
#test one token
clear_state(L_BEGIN, L_CLOSE,lpddr_c,lpddr_h)
run_mamba(L_BEGIN, L_CLOSE, 0, verbose=True, freq=MHz425)
# check result
DUT_Y         = lpddr_y[:NUM_X]

print(REF_Y        [:50])
print(DUT_Y        [:50])

for i in range(NUM_X):
    if REF_Y[i]!=DUT_Y[i] and i<10:
        print(f"error! get{DUT_Y[i]},expect{REF_Y[i]}")

Reading IDLE: 1
L_BEGIN:  0
L_CLOSE:  64
MEMORY_W: 50000000000
MEMORY_X: 50080000000
MEMORY_Y: 50090000000
MEMORY_C: 500a0000000
MEMORY_H: 500b0000000
POS:      0
elapsed time:        0.1416
[  9278  -8385  -9213 -20183  30386  -7689 -43692 -20380  19077  12598
  15228  27512 -23568 -85883   5037  18216  30621  -3336 -22862  36506
  32318 -16406  14831   1978 -14496    198  19377  18132  35752  22985
 -15173  48813  25275 -42848  -4636 -11076  16266 -47724 -47019   9651
 -43688  47630 -35128 -82396 -37622 -53671  21451  29436 -23309    186]
[  9278  -8385  -9213 -20183  30386  -7689 -43692 -20380  19077  12598
  15228  27512 -23568 -85883   5037  18216  30621  -3336 -22862  36506
  32318 -16406  14831   1978 -14496    198  19377  18132  35752  22985
 -15173  48813  25275 -42848  -4636 -11076  16266 -47724 -47019   9651
 -43688  47630 -35128 -82396 -37622 -53671  21451  29436 -23309    186]


In [26]:
# repeat to test power
# ultimate reset
pl_reset.reset()
clock.refresh(MHz425)

# start
mamba.L_BEGIN  = 0
mamba.L_CLOSE  = L_MAX
mamba.MEMORY_W = ADDR_W
mamba.MEMORY_X = ADDR_X
mamba.MEMORY_Y = ADDR_Y
mamba.MEMORY_C = ADDR_C
mamba.MEMORY_H = ADDR_H

test_num=20
total_time = 0
start   = time.time()
for i in range(test_num):
    lpddr_x[:NUM_X] = REF_X
    mamba.T = 1
    # time
    while True:
        sleep(0.000001)
        status = mamba.IDLE
        if mamba.IDLE:
            break
    DUT_Y         = lpddr_y[:NUM_X]
end     = time.time()
elapsed = end - start
total_time += elapsed
avg_time = total_time / test_num
    
print(f"elapsed time: {elapsed: .4f}, average time: {avg_time: .4f}, average throughput: {1.0/avg_time:.4f} token/s")

elapsed time:  2.8311, average time:  0.1416, average throughput: 7.0644 token/s


In [10]:
V=50288
repetition_penalty=1.2
embedding_table = np.fromfile(f"./bin/embedding_int32.bin",  dtype=np.int32).reshape(V,C)
rms_weight = np.fromfile(f"./bin/last_rms_weight_fp16.bin",  dtype=np.float16)
lm_head = np.fromfile(f"./bin/lm_head_fp16.bin",  dtype=np.float16).reshape(V,C).transpose(1,0)
pow_o_res=pow(2.0,10)

In [ ]:
repetition_penalty=1.0
def Embedding(token_id):
    return embedding_table[token_id]
def Last_RMS(result):
    x_fp=(result.astype(np.float64)/pow_o_res)
    x_sqrt=np.sqrt(np.sum(np.square(x_fp))/C)
    return (result*rms_weight/x_sqrt).astype(np.float32)
def LM(rms,prev_output_tokens,seqlen):
    logits=rms@lm_head
    if repetition_penalty!=0:
        for i in range(seqlen):
            if logits[prev_output_tokens[i]]<0:
                logits[prev_output_tokens[i]] *= repetition_penalty
            else:
                logits[prev_output_tokens[i]] /= repetition_penalty
    return logits.argmax(-1)
def prefill(input_ids,seqlen):
    clear_state(L_BEGIN, L_CLOSE,lpddr_c,lpddr_h)
    mamba.L_BEGIN  = L_BEGIN
    mamba.L_CLOSE  = L_CLOSE
    mamba.MEMORY_W = ADDR_W
    mamba.MEMORY_X = ADDR_X
    mamba.MEMORY_Y = ADDR_Y
    mamba.MEMORY_C = ADDR_C
    mamba.MEMORY_H = ADDR_H
    mamba.POS      = 0
    for input_id in input_ids:
#         print("input_id:")
#         print(input_id)
        token=Embedding(input_id)
        lpddr_x[:NUM_X] = token
#         mamba.POS      = 0
        mamba.T = 1
        while True:
            sleep(0.0001)
            if mamba.IDLE:
                break
        result=lpddr_y[:NUM_X]
#         print("prefill result:")
#         print(result        [:10])
    rms=Last_RMS(result)
    sample_token=LM(rms,input_ids,seqlen)
    return sample_token
def decode(decode_id,prev_output_tokens,seqlen):
    token=Embedding(decode_id)
    lpddr_x[:NUM_X] = token
    mamba.T = 1
    while True:
        sleep(0.0001)
        if mamba.IDLE:
            break
    result=lpddr_y[:NUM_X]
    rms=Last_RMS(result)
    sample_token=LM(rms,prev_output_tokens,seqlen)
    return sample_token

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("./models--EleutherAI--gpt-neox-20b/snapshots/c292233c833e336628618a88a648727eb3dff0a7")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [13]:
from IPython.display import clear_output

# 清除当前单元格的输出
clear_output(wait=True)

In [28]:
pl_reset.reset()
clock.refresh(MHz425)
decode_length=100
input_ids=np.array([3220, 5798, 4159, 512, 436, 32463, 4877, 2127, 323, 247, 747, 3448, 1566, 285])
seqlen=input_ids.size
print("Prompt:")
tensor=np.array(input_ids)
tensor = tensor.reshape(1,tensor.shape[0])
print(tokenizer.batch_decode(tensor.tolist())[0])
print("\n")
print("Answer:")
sample_token=prefill(input_ids,seqlen)
output_ids=input_ids.tolist()
output_ids.append(sample_token)
seqlen+=1
for i in range(decode_length-1):
    sample_token=decode(sample_token,output_ids,seqlen)
    clear_output(wait=True)
    print("Prompt:")
    tensor=np.array(input_ids)
    tensor = tensor.reshape(1,tensor.shape[0])
    print(tokenizer.batch_decode(tensor.tolist())[0])
    print("\n")
    print("Answer:")
    output_ids.append(sample_token)
    tensor=np.array(output_ids)
    tensor = tensor.reshape(1,tensor.shape[0])
    print(tokenizer.batch_decode(tensor.tolist())[0])
    seqlen+=1


Prompt:
My cat wrote all this CUDA code for a new language model and


Answer:
My cat wrote all this CUDA code for a new language model and I'm trying to figure out how to use it.

I have no idea what any of that means.<|endoftext|>
I'm not sure if the CUDA code is actually doing anything or if it's just some kind of joke.

The only thing I know about Cuda is that you can't get laid in China without using it, but even then I don’t know why you would want your own language model written by a cat.<|endoftext|>
It’s probably something like


In [11]:
def server_prefill(input_ids,seqlen):
    clear_state(L_BEGIN, L_CLOSE,lpddr_c,lpddr_h)
    mamba.L_BEGIN  = L_BEGIN
    mamba.L_CLOSE  = L_CLOSE
    mamba.MEMORY_W = ADDR_W
    mamba.MEMORY_X = ADDR_X
    mamba.MEMORY_Y = ADDR_Y
    mamba.MEMORY_C = ADDR_C
    mamba.MEMORY_H = ADDR_H
    mamba.POS      = 0
    for input_id in input_ids:
        token=embedding_table[input_id]
        lpddr_x[:NUM_X] = token
        mamba.T = 1
        while True:
            sleep(0.0001)
            if mamba.IDLE:
                break
        result=lpddr_y[:NUM_X]
    return result
def server_decode(decode_id):
    token=embedding_table[decode_id]
    lpddr_x[:NUM_X] = token
    mamba.T = 1
    while True:
        sleep(0.0001)
        if mamba.IDLE:
            break
    result=lpddr_y[:NUM_X]
    return result

In [25]:
import socket
import threading
import sys

def handle_client(client_socket):
    while True:
        try:
            prefill_length_tensor = np.frombuffer(client_socket.recv(4),dtype=np.int32)
            if len(prefill_length_tensor) == 0:
                break
            prefill_length = prefill_length_tensor[0].item()
            pl_reset.reset()
            clock.refresh(MHz425)
            
            data = client_socket.recv(prefill_length*4)
            input_ids = np.frombuffer(data, dtype=np.int32)
        
            decode_length = np.frombuffer(client_socket.recv(4),dtype=np.int32)[0].item()
            response = server_prefill(input_ids,prefill_length)
            response_data = response.tobytes()
           
            for decode_times in range(decode_length):      
                for i in range(4):   
                    data=response_data[i*2560:(i+1)*2560]
                    client_socket.send(data)
                if decode_times!=decode_length-1:
                    token_id_bytes = client_socket.recv(4)
                    token_id = np.frombuffer(token_id_bytes, dtype=np.int32)[0]
                    response = server_decode(token_id)
                    response_data = response.tobytes()
                
            print("success")
        
        except Exception as e:
            print(f"处理客户端消息时发生错误: {e}")
            break
    
    client_socket.close()

def server_input_listener(server):
    """
    检测用户输入的线程
    """
    while True:
        command = input("请输入 'exit' 来关闭服务器: ")
        if command.strip().lower() == 'exit':
            print("正在关闭服务器...")
            server.close()  # 关闭服务器套接字
            sys.exit(0)  # 退出整个程序

def start_server(host, port):
    """
    启动服务器的主函数
    """
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind((host, port))
    server.listen(5)
    print(f"服务器启动，监听 {host}:{port} ...")

    # 启动一个线程来监听用户输入
    input_thread = threading.Thread(target=server_input_listener, args=(server,), daemon=True)
    input_thread.start()

    try:
        while True:
            client_socket, addr = server.accept()
            print(f"与客户端 {addr} 连接成功")
            client_handler = threading.Thread(target=handle_client, args=(client_socket,))
            client_handler.start()
    except Exception as e:
        print(f"服务器错误: {e}")
    finally:
        print("服务器已关闭")

if __name__ == "__main__":
    host = "0.0.0.0"
    port = 9191
    start_server(host, port)

服务器启动，监听 0.0.0.0:9191 ...
与客户端 ('10.129.164.177', 33756) 连接成功
success
与客户端 ('10.129.164.177', 33772) 连接成功
success
与客户端 ('10.129.164.177', 33778) 连接成功
success
与客户端 ('10.129.164.177', 33780) 连接成功
success
与客户端 ('10.129.164.177', 33782) 连接成功
success
与客户端 ('10.129.164.177', 33784) 连接成功
success
与客户端 ('10.129.164.177', 33786) 连接成功
success
与客户端 ('10.129.164.177', 33790) 连接成功
success
与客户端 ('10.129.164.177', 33792) 连接成功
success
与客户端 ('10.129.164.177', 33794) 连接成功
success
与客户端 ('10.129.164.177', 33800) 连接成功
success
请输入 'exit' 来关闭服务器: exit
正在关闭服务器...
服务器已关闭


KeyboardInterrupt: 